# Workspace Access Report with Temporal Tracking

This notebook retrieves all Fabric workspace access permissions and tracks when access was granted or revoked.

**Features:**
- Expands group memberships to show individual users
- Tracks `granted_on_dt` for when access was first detected
- Tracks `revoked_on_dt` for when access was removed
- Maintains full history in the lakehouse table

**Usage:** Run this notebook daily to maintain accurate temporal tracking.

In [ ]:
import requests
import pandas as pd
from typing import List, Dict
from datetime import datetime
import json

# ---------------------------------------------------------------------------
# Optional service-principal fallback credentials.
# Leave empty to rely on notebookutils (interactive / workspace identity).
# When running via a Fabric pipeline, inject these as pipeline parameters.
# The SP needs at minimum: GroupMember.Read.All (or Group.Read.All) on Graph.
# ---------------------------------------------------------------------------
SP_CLIENT_ID     = ""  # set via pipeline parameter or notebookutils.widgets.get('sp_client_id')
SP_CLIENT_SECRET = ""  # set via pipeline parameter or notebookutils.widgets.get('sp_client_secret')
SP_TENANT_ID     = ""  # set via pipeline parameter or notebookutils.widgets.get('sp_tenant_id')

def get_fabric_token():
    """Get Fabric API access token"""
    return notebookutils.credentials.getToken("https://analysis.windows.net/powerbi/api")

def get_graph_token_via_service_principal():
    """
    Fallback: acquire a Graph token using a service principal via MSAL.
    Reads SP_CLIENT_ID / SP_CLIENT_SECRET / SP_TENANT_ID from the variables
    defined at the top of this cell (set them via pipeline parameters or
    notebookutils.widgets / Key Vault linked service).
    """
    try:
        import msal
        print("  🔄 Fallback: acquiring Graph token via service principal (MSAL)...")
        if not all([SP_CLIENT_ID, SP_CLIENT_SECRET, SP_TENANT_ID]):
            raise ValueError(
                "SP_CLIENT_ID, SP_CLIENT_SECRET, and SP_TENANT_ID must all be set. "
                "Assign them at the top of the auth cell or inject via pipeline parameters."
            )
        app = msal.ConfidentialClientApplication(
            SP_CLIENT_ID,
            authority=f"https://login.microsoftonline.com/{SP_TENANT_ID}",
            client_credential=SP_CLIENT_SECRET,
        )
        result = app.acquire_token_for_client(
            scopes=["https://graph.microsoft.com/.default"]
        )
        if "access_token" in result:
            print("  ✓ Graph token acquired via service principal")
            return result["access_token"]
        raise Exception(result.get("error_description") or result.get("error") or str(result))
    except Exception as e:
        print(f"  ❌ Service principal fallback failed: {e}")
        raise

def get_graph_token():
    """Get a Microsoft Graph token via service principal (MSAL)."""
    return get_graph_token_via_service_principal()

# Get tokens
print("🔐 Authenticating...")
fabric_token = get_fabric_token()
print("✓ Fabric token acquired")

graph_token = get_graph_token()

today = datetime.now().date().isoformat()

print(f"\n✓ Authentication complete")
print(f"✓ Snapshot date: {today}")
print(f"✓ Group expansion enabled")

In [ ]:
def get_workspaces(token: str) -> List[Dict]:
    """Retrieve all Fabric workspaces"""
    url = "https://api.fabric.microsoft.com/v1/workspaces"
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    return response.json().get('value', [])

# Get all workspaces
workspaces = get_workspaces(fabric_token)
print(f"✓ Found {len(workspaces)} workspaces")

In [ ]:
def get_workspace_role_assignments(workspace_id: str, token: str) -> List[Dict]:
    """Get role assignments for a specific workspace"""
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/roleAssignments"
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    return response.json().get('value', [])

print("✓ Workspace role assignment function ready")

In [ ]:
def get_group_members(group_id: str, token: str) -> List[Dict]:
    """Get all members of a security group (handles pagination)"""
    members = []
    url = f"https://graph.microsoft.com/v1.0/groups/{group_id}/transitiveMembers?$select=id,displayName,userPrincipalName,mail"
    headers = {"Authorization": f"Bearer {token}"}
    
    while url:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        members.extend(data.get('value', []))
        url = data.get('@odata.nextLink')  # Handle pagination
    
    return members

print("✓ Group member expansion function ready")

In [ ]:
# Collect all access data
all_access_records = []

for workspace in workspaces:
    workspace_id = workspace['id']
    workspace_name = workspace['displayName']
    
    try:
        role_assignments = get_workspace_role_assignments(workspace_id, fabric_token)
        
        for assignment in role_assignments:
            principal_type = assignment.get('principal', {}).get('type')
            
            if principal_type == 'Group':
                # Expand group to individual users
                group_id = assignment['principal']['id']
                group_name = assignment['principal']['displayName']
                
                try:
                    members = get_group_members(group_id, graph_token)
                    for member in members:
                        if member.get('@odata.type') == '#microsoft.graph.user':
                            all_access_records.append({
                                'WorkspaceId': workspace_id,
                                'WorkspaceName': workspace_name,
                                'PrincipalType': 'User',
                                'PrincipalId': member['id'],
                                'PrincipalDisplayName': member.get('displayName', ''),
                                'Role': assignment['role'],
                                'GroupName': group_name,
                                'GroupId': group_id,
                                'snapshot_date': today
                            })
                except Exception as e:
                    print(f"  ⚠️ Could not expand group {group_name}: {e}")
            else:
                # Direct user/service principal assignment
                all_access_records.append({
                    'WorkspaceId': workspace_id,
                    'WorkspaceName': workspace_name,
                    'PrincipalType': principal_type,
                    'PrincipalId': assignment['principal']['id'],
                    'PrincipalDisplayName': assignment['principal'].get('displayName', ''),
                    'Role': assignment['role'],
                    'GroupName': None,
                    'GroupId': None,
                    'snapshot_date': today
                })
    except Exception as e:
        print(f"⚠️ Error processing workspace {workspace_name}: {e}")

print(f"\n✓ Collected {len(all_access_records)} access records")

In [ ]:
# Convert to DataFrame
current_snapshot = pd.DataFrame(all_access_records)

# Create unique access key for change detection
def create_access_key(row):
    return f"{row['WorkspaceId']}|{row['PrincipalId']}|{row['Role']}"

current_snapshot['access_key'] = current_snapshot.apply(create_access_key, axis=1)

print(f"✓ Created access keys for {len(current_snapshot)} records")
current_snapshot.head(10)

In [ ]:
# Read historical data from lakehouse (if exists)
try:
    historical_df = spark.read.format("delta").table("workspace_access_report")
    historical_df = historical_df.toPandas()
    print(f"✓ Found {len(historical_df)} historical records")
    
    # Filter to only active records (not revoked)
    active_historical = historical_df[historical_df['revoked_on_dt'].isna()].copy()
    print(f"✓ {len(active_historical)} active historical records")
except Exception as e:
    print(f"ℹ️ No historical data found (this is normal on first run): {e}")
    active_historical = pd.DataFrame()

In [ ]:
# Detect changes
if not active_historical.empty:
    current_keys = set(current_snapshot['access_key'])
    historical_keys = set(active_historical['access_key'])
    
    new_access = current_keys - historical_keys
    removed_access = historical_keys - current_keys
    continuing_access = current_keys & historical_keys
    
    print(f"\n📊 Change Summary:")
    print(f"  ✅ New access granted: {len(new_access)}")
    print(f"  ➡️ Continuing access: {len(continuing_access)}")
    print(f"  ❌ Access revoked: {len(removed_access)}")
else:
    print("ℹ️ No historical data - all current access will be marked as new")
    new_access = set(current_snapshot['access_key'])
    removed_access = set()
    continuing_access = set()

In [ ]:
# Set timestamps for new and revoked access
def set_timestamps(row):
    if row['access_key'] in new_access:
        row['granted_on_dt'] = today
        row['revoked_on_dt'] = None
    elif not active_historical.empty:
        # Get the original granted_on_dt from historical data
        hist_row = active_historical[active_historical['access_key'] == row['access_key']]
        if not hist_row.empty:
            row['granted_on_dt'] = hist_row.iloc[0]['granted_on_dt']
            row['revoked_on_dt'] = None
    else:
        row['granted_on_dt'] = today
        row['revoked_on_dt'] = None
    return row

current_snapshot = current_snapshot.apply(set_timestamps, axis=1)

# Handle revoked access
if not active_historical.empty and removed_access:
    revoked_records = active_historical[active_historical['access_key'].isin(removed_access)].copy()
    revoked_records['revoked_on_dt'] = today
    revoked_records['snapshot_date'] = today
    
    # Combine current + revoked
    final_df = pd.concat([current_snapshot, revoked_records], ignore_index=True)
else:
    final_df = current_snapshot

print(f"✓ Timestamps set for {len(final_df)} records")

In [ ]:
# Convert to Spark DataFrame and save to Delta table
spark_df = spark.createDataFrame(final_df)
spark_df.write.format("delta").mode("append").saveAsTable("workspace_access_report")

print(f"\n✅ Data saved to lakehouse table: workspace_access_report")
print(f"   Records written: {len(final_df)}")

In [ ]:
# Display summary statistics
print("\n📈 Summary Statistics:")
print(f"Total workspaces scanned: {len(workspaces)}")
print(f"Total access records: {len(current_snapshot)}")
print(f"\nRole breakdown:")
print(current_snapshot['Role'].value_counts())
print(f"\nTop 10 workspaces by access count:")
print(current_snapshot.groupby('WorkspaceName').size().sort_values(ascending=False).head(10))